In [104]:
# Silo Data into Diagnostic, Misc, and Unlabeled Sets

import os
import pandas
from sklearn.model_selection import train_test_split
from dotenv import load_dotenv

In [105]:
load_dotenv()

features_path = os.getenv("FEATURES_PATH")
gexml_samplesheet_path = os.getenv("GEXML_SAMPLESHEET_PATH")
output_dir = os.getenv("OUTPUT_DIR")
min_samples_per_class = int(os.getenv("MIN_SAMPLES_PER_CLASS", 5))

print("Configuration:")
print(f"Features path: {features_path}")
print(f"Samplesheet path: {gexml_samplesheet_path}")
print(f"Output directory: {output_dir}")
print(f"Minimum samples per class: {min_samples_per_class}")

Configuration:
Features path: /home/mariya/Data/pallas/nf-core-gexml-output/transpose/transposed.csv
Samplesheet path: /home/mariya/Data/pallas/samplesheet.csv
Output directory: /home/mariya/Data/pallas/training
Minimum samples per class: 5


In [106]:
# Make sure output dir exists
os.makedirs(output_dir, exist_ok=True)

# Function to save data
def save_data_silo(X, y, prefix):
    X.to_csv(f'{output_dir}/pallas.{prefix}.X.csv')
    y.to_csv(f'{output_dir}/pallas.{prefix}.y.tsv', sep='\t')

In [107]:
if not gexml_samplesheet_path.endswith('.csv'):
    raise ValueError("Target file must be in .csv format.")
target = pandas.read_csv(
    gexml_samplesheet_path, sep=",", header=0, index_col=0)

target.index.name = "id"
target = target[['subtype']].rename(columns={'subtype': 'y'})
target

,y
id,
SJHBBBM4X52,"Disease,B-ALL,ETV6::RUNX1-group,ETV6::RUNX1"
SJHBBBM4JQZ,"Disease,B-ALL,ETV6::RUNX1-group,ETV6::RUNX1"
SJHBBBM5SGJ,"Disease,B-ALL,ETV6::RUNX1-group,ETV6::RUNX1"
SJHBBBM5TXY,"Disease,B-ALL,ETV6::RUNX1-group,ETV6::RUNX1"
SJHBBBM2GV5,"Disease,B-ALL,ETV6::RUNX1-group,ETV6::RUNX1"
...,...
SJBALL020571,"Disease,B-ALL,ZNF384-r"
SJBALL020779,"Disease,B-ALL,BCR::ABL1-group,BCR::ABL1-like"
SJBALL020946,"Disease,B-ALL,aneuploidy,high hyperdiploid"


In [108]:
features = pandas.read_csv(features_path, header=0, index_col=0)
data = features.join(target)
data

,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000293661,ENSG00000293662,ENSG00000293663,ENSG00000293685,ENSG00000293689,ENSG00000293691,ENSG00000300510,ENSG00000310517,ENSG00000310525,y
id,,,,,,,,,,,,,,,,,,,,,
16-001,-1.030936,-1.324478,4.107419,4.481528,5.313913,4.318306,-0.006233,0.682168,4.419238,6.533335,...,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,"Disease,B-ALL,ETV6::RUNX1-group,ETV6::RUNX1"
16-002,-0.683009,-1.324478,4.564085,2.704030,4.879351,3.515584,-0.048009,2.102919,3.897675,6.750946,...,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,"Disease,B-ALL,ETV6::RUNX1-group,ETV6::RUNX1"
16-003,-0.795420,-1.324478,4.001798,4.804804,5.278272,2.338792,-0.009478,1.674171,4.069893,6.442101,...,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,"Disease,B-ALL,ETV6::RUNX1-group,ETV6::RUNX1"
16-006,-0.976469,-1.324478,4.335444,4.571098,4.364741,3.808390,-1.051959,1.764196,3.514185,6.925137,...,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,"Disease,B-ALL,ETV6::RUNX1-group,ETV6::RUNX1"
16-008,3.986329,-1.324478,4.179535,5.059009,4.009802,5.316485,-0.391770,2.303854,5.361421,7.219616,...,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,"Disease,T-ALL"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SJTALL169_D,-1.324478,-1.324478,5.020547,3.936178,3.989832,4.000849,2.358342,5.997673,6.300551,5.633695,...,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,"Disease,T-ALL"
SJTALL174_D,-1.096536,-1.324478,3.893166,4.396042,4.976536,5.103507,5.563309,4.615500,4.329157,7.725124,...,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,"Disease,T-ALL"
SJTALL182_D,-1.324478,-1.324478,3.926871,4.020709,6.216572,4.546595,2.511116,4.795256,5.371875,6.776146,...,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,"Disease,T-ALL"


In [109]:
unlabeled = data[data['y'].str.contains("B-other|AML-other|ALLNOS")]
unlabeled['y']

id
16-022             Disease,B-ALL,B-other
16-027             Disease,B-ALL,B-other
16-029             Disease,B-ALL,B-other
16-033             Disease,B-ALL,B-other
16-037             Disease,B-ALL,B-other
                           ...          
SJHM031652_D1       UNRECOGNIZED: ALLNOS
SJHM031965_D1      Disease,AML,AML-other
SJINF071_D         Disease,B-ALL,B-other
SJMLL030007_D1     Disease,AML,AML-other
SJTALL030741_D1     UNRECOGNIZED: ALLNOS
Name: y, Length: 575, dtype: object

In [110]:
save_data_silo(unlabeled.drop(columns=['y']), unlabeled['y'], 'unlabeled')

In [111]:
multiclass = data[data['y'].str.contains("MULTICLASS")]
# Exclude all that are also in unlabeled
multiclass = multiclass[~multiclass.index.isin(unlabeled.index)]
multiclass['y']

id
16-044              UNRECOGNIZED: MULTICLASS: PAX5_alteration,BCR-...
16-099              UNRECOGNIZED: MULTICLASS: KMT2A-R,PAX5_alteration
16-139                    UNRECOGNIZED: MULTICLASS: ETV6-RUNX1,iAMP21
16-162                 UNRECOGNIZED: MULTICLASS: HHD,BCR-ABL1_or_like
16-221                       UNRECOGNIZED: MULTICLASS: ETV6-RUNX1,HHD
                                          ...                        
SJMDS005142_D2      UNRECOGNIZED: MULTICLASS: Primary_Diagnosis:AL...
SJMDS040265_D2      UNRECOGNIZED: MULTICLASS: Primary_Diagnosis:AL...
SJPHALL020034_D1    UNRECOGNIZED: MULTICLASS: Lineage:B,Primary_su...
SJPHALL020036_D1    UNRECOGNIZED: MULTICLASS: Lineage:B,Primary_su...
SJPHALL020043_D1    UNRECOGNIZED: MULTICLASS: Lineage:B,Primary_su...
Name: y, Length: 176, dtype: object

In [112]:
save_data_silo(multiclass.drop(columns=['y']), multiclass['y'], 'multiclass')

In [113]:
unrecognized = data[data['y'].str.startswith("UNRECOGNIZED")]
# Exclude all that are also in multiclass or unlabeled
unrecognized = unrecognized[~unrecognized.index.isin(unlabeled.index)]
unrecognized = unrecognized[~unrecognized.index.isin(multiclass.index)]
unrecognized['y']

id
PARAPE                     UNRECOGNIZED: CRLF2(non-Ph-like)
PARSLL                     UNRECOGNIZED: CRLF2(non-Ph-like)
PASIZE                     UNRECOGNIZED: CRLF2(non-Ph-like)
SJALL040069                UNRECOGNIZED: CRLF2(non-Ph-like)
SJBALL004200               UNRECOGNIZED: CRLF2(non-Ph-like)
                                         ...               
SJHM033701_D1                            UNRECOGNIZED: AMKL
SJHM034159_D1                        UNRECOGNIZED: MPALBNOS
SJTALL030047_R1_RELAPSE                  UNRECOGNIZED: ALAL
SJTALL033119_D1                      UNRECOGNIZED: MPALTNOS
SJTALL033883_D1                      UNRECOGNIZED: MPALTNOS
Name: y, Length: 244, dtype: object

In [114]:
save_data_silo(unrecognized.drop(columns=['y']), unrecognized['y'], 'unrecognized')

In [115]:
relapse = data[data.index.str.match(
    r'^(SJ.*_R\d*|SJ.*_RELAPSE|ALL.*_(2|3))$')]
relapse = relapse[~relapse.index.isin(unlabeled.index)]
relapse = relapse[~relapse.index.isin(multiclass.index)]
relapse = relapse[~relapse.index.isin(unrecognized.index)]
relapse['y']

id
ALL_317_2                             Disease,B-ALL,aneuploidy,iAMP21
ALL_317_3                             Disease,B-ALL,aneuploidy,iAMP21
ALL_555_2                  Disease,B-ALL,aneuploidy,high hyperdiploid
ALL_680_2                                       Disease,B-ALL,PAX5alt
SJAML001409_R1_RELAPSE                            Disease,AML,NUP98-r
                                              ...                    
SJINF060_R_RELAPSE                              Disease,B-ALL,KMT2A-r
SJINF061_R_RELAPSE                                      Disease,T-ALL
SJINF065_R_RELAPSE                              Disease,B-ALL,KMT2A-r
SJINF073_R_RELAPSE                              Disease,B-ALL,KMT2A-r
SJTALL031014_R1_RELAPSE                                 Disease,T-ALL
Name: y, Length: 107, dtype: object

In [116]:
relapse


,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000293661,ENSG00000293662,ENSG00000293663,ENSG00000293685,ENSG00000293689,ENSG00000293691,ENSG00000300510,ENSG00000310517,ENSG00000310525,y
id,,,,,,,,,,,,,,,,,,,,,
ALL_317_2,-1.324478,-1.324478,4.915263,2.945381,2.564653,1.699018,-0.928756,0.363519,4.967439,6.695798,...,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,"Disease,B-ALL,aneuploidy,iAMP21"
ALL_317_3,0.651845,-1.324478,4.530764,2.864210,2.296835,2.514003,0.749965,1.898208,5.268416,6.300037,...,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,"Disease,B-ALL,aneuploidy,iAMP21"
ALL_555_2,-0.531369,-1.324478,5.603671,4.476163,4.911468,1.781037,-0.301808,1.971610,5.373838,7.307683,...,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,"Disease,B-ALL,aneuploidy,high hyperdiploid"
ALL_680_2,-1.206567,-1.324478,4.396281,4.116114,4.824950,2.680678,-0.809372,2.967722,4.105941,6.760152,...,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,"Disease,B-ALL,PAX5alt"
SJAML001409_R1_RELAPSE,-1.087563,-1.324478,4.293492,6.052864,4.161243,0.385248,5.901124,5.605057,4.439255,6.305405,...,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,"Disease,AML,NUP98-r"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SJINF060_R_RELAPSE,-0.906520,-1.324478,3.381947,4.688611,4.860938,7.371245,3.339518,5.176061,5.275450,6.644647,...,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,"Disease,B-ALL,KMT2A-r"
SJINF061_R_RELAPSE,2.898651,-0.770947,3.384411,4.677358,5.376275,8.507647,4.472619,4.794265,7.221556,6.005271,...,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,"Disease,T-ALL"
SJINF065_R_RELAPSE,-0.684880,-1.324478,3.274038,4.660292,4.988899,7.440936,3.639846,4.668496,5.887056,6.516899,...,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,-1.324478,"Disease,B-ALL,KMT2A-r"


In [117]:
save_data_silo(relapse.drop(columns=['y']), relapse['y'], 'relapse')

In [118]:
replicate = data[data.index.str.match(r'^ALL_.*_r$')]
replicate['y']

id
ALL_1026_r                         Disease,B-ALL,PAX5alt
ALL_210_r     Disease,B-ALL,aneuploidy,high hyperdiploid
ALL_257_r                         Disease,B-ALL,ZNF384-r
ALL_358_r                                  Disease,T-ALL
ALL_368_r                          Disease,B-ALL,PAX5alt
ALL_380_r     Disease,B-ALL,aneuploidy,high hyperdiploid
ALL_464_r     Disease,B-ALL,aneuploidy,high hyperdiploid
ALL_680_r                          Disease,B-ALL,PAX5alt
Name: y, dtype: object

In [119]:
save_data_silo(replicate.drop(columns=['y']), replicate['y'], 'replicate')

In [120]:
rare = data[data['y'].map(data['y'].value_counts()) < min_samples_per_class]
# Exclude everything that is already in other silos
rare = rare[~rare.index.isin(unlabeled.index)]
rare = rare[~rare.index.isin(multiclass.index)]
rare = rare[~rare.index.isin(unrecognized.index)]
rare = rare[~rare.index.isin(relapse.index)]
rare = rare[~rare.index.isin(replicate.index)]
rare['y']

id
SJAML031264_D1            Disease,AML,MECOM-r
SJAML033572_D1            Disease,AML,KAT6A-r
SJAML033585_D1     Disease,AML,PICALM::MLLT10
SJAML033710_D1            Disease,AML,MECOM-r
SJAML040281_D3            Disease,AML,MECOM-r
SJAML060296_D1            Disease,AML,KAT6A-r
SJAML061501_D2     Disease,AML,PICALM::MLLT10
SJAML074161_D1     Disease,AML,PICALM::MLLT10
SJBALL020977_D1      Disease,B-ALL,IGH::CEBPD
SJBALL030327_D1      Disease,B-ALL,IGH::CEBPD
SJBALL030519_D1      Disease,B-ALL,IGH::CEBPD
SJBALL033673_D1      Disease,B-ALL,IGH::CEBPD
Name: y, dtype: object

In [121]:
save_data_silo(rare.drop(columns=['y']), rare['y'], 'rare')

In [122]:
labeled_diagnostic = data[~data.index.isin(unlabeled.index)]
labeled_diagnostic = labeled_diagnostic[~labeled_diagnostic.index.isin(multiclass.index)]
labeled_diagnostic = labeled_diagnostic[~labeled_diagnostic.index.isin(unrecognized.index)]
labeled_diagnostic = labeled_diagnostic[~labeled_diagnostic.index.isin(relapse.index)]
labeled_diagnostic = labeled_diagnostic[~labeled_diagnostic.index.isin(replicate.index)]
labeled_diagnostic = labeled_diagnostic[~labeled_diagnostic.index.isin(rare.index)]
labeled_diagnostic['y']

id
16-001         Disease,B-ALL,ETV6::RUNX1-group,ETV6::RUNX1
16-002         Disease,B-ALL,ETV6::RUNX1-group,ETV6::RUNX1
16-003         Disease,B-ALL,ETV6::RUNX1-group,ETV6::RUNX1
16-006         Disease,B-ALL,ETV6::RUNX1-group,ETV6::RUNX1
16-008                                       Disease,T-ALL
                                  ...                     
SJTALL169_D                                  Disease,T-ALL
SJTALL174_D                                  Disease,T-ALL
SJTALL182_D                                  Disease,T-ALL
SJTALL186_D                                  Disease,T-ALL
SJTALL187_D                                  Disease,T-ALL
Name: y, Length: 3191, dtype: object

In [123]:
save_data_silo(labeled_diagnostic.drop(columns=['y']), labeled_diagnostic['y'], 'labeled_diagnostic')

In [124]:
# Add up lengths of all silos and ensure it matches original data length
total_length = (len(unlabeled) + len(multiclass) + len(unrecognized) +
                len(relapse) + len(replicate) + len(rare) + len(labeled_diagnostic))
total_length

assert total_length == len(data), "Silos do not add up to original data length!"